In [1]:
# read in LIST.
# list all ICU dates (CAM) for each subject
# locate corresponding fileID for each subject. init =0, if not found 'NotFound'
    # 1 column: converted: 1/0
    # 1 column: notes: string notes.

before running this code and update ICUSleep_DataTable, two things/codes need to be run:  
- StepA1_bedmaster ... : ADT and bedmaster data from cdac database needs to be updated for new studyIDs, save ADT and bedmaster linked data
- FindNonConverted ... : merge info from StepA1 and check which of the bedmaster files have been converted (i.e. check if files exist in converted directory)

first time initialize list, net time change it (not only appending, since conversion status can change etc.)  
But: code appends new subjects. i.e. if subject 999 is not in list and gets read in with LIST and CAM Dates, its information gets appended to the table (that means if you want to completly update information about a subject, it needs to be deleted from the table first)

In [2]:
import pandas as pd
import numpy as np
import os
from datetime import timedelta

In [3]:
GitHubBackupDir = './BackupFiles/'

#### Initialization of List. Needs to be run only at first time.

In [4]:
new_initialization = 0
ReportsDirTemp = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/ExportedReports/'
# LogDir = 'C:/Users/wg984/Wolfgang/ICU-Sleep/'
LogDir = '//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/Study/'

if new_initialization:
    ICUSleep_DataTable = pd.DataFrame(columns = ['StudyID','CAM_Date','BMFileConverted','BMDataAllGood','BMFileID','BMFileNotes','AirGoDone','AirGoFileNotes','EventName','ICU_Site','ICU_Bed','MRN','CAM_performed'])
else:
    ICUSleep_DataTable = pd.read_csv(os.path.join(LogDir,'ICUSleep_DataTable.csv'))
    ICUSleep_DataTable.CAM_Date = pd.to_datetime(ICUSleep_DataTable.CAM_Date, infer_datetime_format=1)

In [5]:
ICUSleep_DataTable.head()

,StudyID,ADT_TransferIn,ADT_TransferOut,CAM_Date,BMFileConverted,BMDataAllGood,BMFileID,BMFileNotes,AirGoDone,AirGoFileNotes,EventName,ICU_Site,ICU_Bed,MRN,CAM_performed,ADT_Admission,ADT_Discharge,ADT_Bed
0,1,2018-06-05 07:16:00,2018-06-08 14:45:00,2018-06-06 16:51:00,1.0,NaN,BLK07_768-1528197453,NaN,NaN,NaN,"ICU Day 1, pm",Blake 7,68,1298742,Yes,2018-06-04 21:13:00,2018-06-10 14:01:00,B0768 A
1,1,2018-06-05 07:16:00,2018-06-08 14:45:00,2018-06-07 10:30:00,1.0,NaN,BLK07_768-1528197453,NaN,NaN,NaN,"ICU Day 2, am",Blake 7,68,1298742,Yes,2018-06-04 21:13:00,2018-06-10 14:01:00,B0768 A
2,1,2018-06-05 07:16:00,2018-06-08 14:45:00,2018-06-07 17:30:00,1.0,NaN,BLK07_768-1528197453,NaN,NaN,NaN,"ICU Day 2, pm",Blake 7,68,1298742,Yes,2018-06-04 21:13:00,2018-06-10 14:01:00,B0768 A
3,1,2018-06-05 07:16:00,2018-06-08 14:45:00,2018-06-08 11:07:00,1.0,NaN,BLK07_768-1528197453,NaN,NaN,NaN,"ICU Day 3, am",Blake 7,68,1298742,Yes,2018-06-04 21:13:00,2018-06-10 14:01:00,B0768 A
4,1,NaN,NaN,2018-06-08 17:19:00,NaN,NaN,BLK07_768-1528197453,NaN,NaN,NaN,"Floor Day 1, pm",NaN,NaN,1298742,Yes,NaN,NaN,NaN


In [6]:
ICUSleep_DataTable.tail()

,StudyID,ADT_TransferIn,ADT_TransferOut,CAM_Date,BMFileConverted,BMDataAllGood,BMFileID,BMFileNotes,AirGoDone,AirGoFileNotes,EventName,ICU_Site,ICU_Bed,MRN,CAM_performed,ADT_Admission,ADT_Discharge,ADT_Bed
2693,194,NaN,NaN,2019-12-01 09:57:00,NaN,NaN,BLK12_1290-1574722511,NaN,NaN,NaN,"Floor Day 4, am",NaN,NaN,6175987,Yes,NaN,NaN,NaN
2694,194,NaN,NaN,2019-12-01 18:00:00,NaN,NaN,BLK12_1290-1574722511,NaN,NaN,NaN,"Floor Day 4, pm",NaN,NaN,6175987,Yes,NaN,NaN,NaN
2695,194,NaN,NaN,2019-12-02 10:15:00,NaN,NaN,BLK12_1290-1574722511,NaN,NaN,NaN,"Floor Day 5, am",NaN,NaN,6175987,Yes,NaN,NaN,NaN
2696,194,NaN,NaN,2019-12-02 17:40:00,NaN,NaN,BLK12_1290-1574722511,NaN,NaN,NaN,"Floor Day 5, pm",NaN,NaN,6175987,"No, patient unavailable",NaN,NaN,NaN
2697,194,NaN,NaN,2019-12-03 18:26:00,NaN,NaN,BLK12_1290-1574722511,NaN,NaN,NaN,"Floor Day 6, pm",NaN,NaN,6175987,"No, patient refused",NaN,NaN,NaN


In [7]:
LIST = pd.read_csv( os.path.join(ReportsDirTemp,'LIST.csv') )
CAM_Date = pd.read_csv( os.path.join(ReportsDirTemp,'CAM_Date.csv') )

In [8]:
LIST = LIST.drop(['Event Name','Repeat Instrument', 'Repeat Instance','First Day Enrolled in Study:',
       'First Name:', 'Last Name:'], axis = 1)

In [9]:
LIST.columns = ['StudyID','ICU_Site','ICU_Bed','MRN']
LIST = LIST.mask(LIST.StudyID=='98a').dropna()
LIST.StudyID = LIST.StudyID.astype('int')
LIST.MRN = LIST.MRN.astype('int')
# LIST = LIST.dropna(how='any')
LIST.tail()

,StudyID,ICU_Site,ICU_Bed,MRN
190,190,Ellison 4,414,1742644
191,191,Blake 7,56,1983461
192,192,Blake 12,72,1750989
193,193,Blake 12,52,4533154
194,194,Blake 12,90,6175987


In [10]:
CAM_Date.columns = CAM_Date.columns.str.strip()

In [11]:
CAM_Date = CAM_Date.drop(['Repeat Instrument','Repeat Instance','Record Created by', 'Note', 'CAM & CAM-ICU evaluated by', 'Complete?'],axis=1)
CAM_Date.columns = ['StudyID','EventName','CAM_performed','CAM_Date']
CAM_Date = CAM_Date[['StudyID','EventName','CAM_Date','CAM_performed']]
CAM_Date = CAM_Date.mask(CAM_Date.StudyID=='98a').dropna()
CAM_Date.StudyID = CAM_Date.StudyID.astype('int')

In [12]:
CAM_Date.CAM_Date = pd.to_datetime(CAM_Date.CAM_Date, infer_datetime_format=1)

In [13]:
print(CAM_Date.shape)
CAM_Date = CAM_Date.dropna()
CAM_Date.tail()

(2680, 4)


,StudyID,EventName,CAM_Date,CAM_performed
2831,194,"Floor Day 5, am",2019-12-02 10:15:00,Yes
2832,194,"Floor Day 5, pm",2019-12-02 17:40:00,"No, patient unavailable"
2834,194,"Floor Day 6, pm",2019-12-03 18:26:00,"No, patient refused"
2918,198,"ICU Day 1, pm",2019-12-02 17:02:00,Yes
2919,198,"ICU Day 2, am",2019-12-03 10:40:00,Yes


In [14]:
ICUSleep_DataTable[ICUSleep_DataTable.StudyID == 53]

,StudyID,ADT_TransferIn,ADT_TransferOut,CAM_Date,BMFileConverted,BMDataAllGood,BMFileID,BMFileNotes,AirGoDone,AirGoFileNotes,EventName,ICU_Site,ICU_Bed,MRN,CAM_performed,ADT_Admission,ADT_Discharge,ADT_Bed
783,53,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,"ICU Day 1, pm",Blake 12,72,2021109,"No, Other",NaN,NaN,NaN


#### STEP 1: 
- append CAM_Date info ('StudyID','EventName','CAM_Date','CAM_performed') to ICUSleep_Table.  
(check if subjectID & CAM_Date pair is already in DataTable and add data if it is not)
- append LIST info

In [15]:
for rowIdx in range(CAM_Date.shape[0]):
#     rowIdx = 0
    StudyID_Tmp = CAM_Date['StudyID'].iloc[rowIdx]
    if StudyID_Tmp == '98a': print('skip MRN 98a'); continue;
    StudyID_Tmp = int(StudyID_Tmp)
    Date_Tmp = CAM_Date['CAM_Date'].iloc[rowIdx]

    SelectionDataTable = ICUSleep_DataTable[(ICUSleep_DataTable.StudyID == StudyID_Tmp) & (ICUSleep_DataTable.CAM_Date == Date_Tmp)]
#     print(SelectionDataTable)
    isNewData = SelectionDataTable.shape[0] == 0 #i.e. nothing is found in selection.

    if isNewData:
#         print('append data for studyID: ' + str(StudyID_Tmp))
        try:
        # collect information to add:
            CAM_selection = CAM_Date.iloc[rowIdx].copy()
            LIST_selection = LIST.loc[LIST.StudyID == StudyID_Tmp].copy()
            if LIST_selection.shape[0]>=1:
                NewData = CAM_selection.copy()
                NewData['MRN'] = LIST_selection.MRN.values[0]
                if 'ICU' in CAM_selection['EventName'] :
                    NewData['ICU_Site'] = LIST_selection.ICU_Site.values[0]
                    NewData['ICU_Bed'] = LIST_selection.ICU_Bed.values[0]
                
                ICUSleep_DataTable = ICUSleep_DataTable.append(NewData)
                
        except:
            print('some error for studyID:' + str(StudyID_Tmp))
            continue
        
        
ICUSleep_DataTable = ICUSleep_DataTable.sort_values(by=['StudyID','CAM_Date'])
        
# ICUSleep_DataTable['CAM_Date'] = pd.to_datetime(ICUSleep_DataTable['CAM_Date'], infer_datetime_format=1)

In [16]:
ICUSleep_DataTable.ADT_TransferIn = pd.to_datetime(ICUSleep_DataTable.ADT_TransferIn, infer_datetime_format=1)
ICUSleep_DataTable.ADT_TransferOut = pd.to_datetime(ICUSleep_DataTable.ADT_TransferOut, infer_datetime_format=1)

ICUSleep_DataTable.CAM_Date = pd.to_datetime(ICUSleep_DataTable.CAM_Date, infer_datetime_format=1)

ICUSleep_DataTable.ADT_Admission = pd.to_datetime(ICUSleep_DataTable.ADT_Admission, infer_datetime_format=1)
ICUSleep_DataTable.ADT_Discharge = pd.to_datetime(ICUSleep_DataTable.ADT_Discharge, infer_datetime_format=1)


##### manually added to list:
studyID 53, fileID: BLK12_1272-1552982319 # does not get appended in current code because no CAM was performed.  
studyID 70, fileID: ELL04_426-1554971651 # does not get appended in current code because no CAM was performed.

In [17]:
ICUSleep_DataTable.tail()

,StudyID,ADT_TransferIn,ADT_TransferOut,CAM_Date,BMFileConverted,BMDataAllGood,BMFileID,BMFileNotes,AirGoDone,AirGoFileNotes,EventName,ICU_Site,ICU_Bed,MRN,CAM_performed,ADT_Admission,ADT_Discharge,ADT_Bed
2693,194,NaT,NaT,2019-12-01 09:57:00,NaN,NaN,BLK12_1290-1574722511,NaN,NaN,NaN,"Floor Day 4, am",NaN,NaN,6175987,Yes,NaT,NaT,NaN
2694,194,NaT,NaT,2019-12-01 18:00:00,NaN,NaN,BLK12_1290-1574722511,NaN,NaN,NaN,"Floor Day 4, pm",NaN,NaN,6175987,Yes,NaT,NaT,NaN
2695,194,NaT,NaT,2019-12-02 10:15:00,NaN,NaN,BLK12_1290-1574722511,NaN,NaN,NaN,"Floor Day 5, am",NaN,NaN,6175987,Yes,NaT,NaT,NaN
2696,194,NaT,NaT,2019-12-02 17:40:00,NaN,NaN,BLK12_1290-1574722511,NaN,NaN,NaN,"Floor Day 5, pm",NaN,NaN,6175987,"No, patient unavailable",NaT,NaT,NaN
2697,194,NaT,NaT,2019-12-03 18:26:00,NaN,NaN,BLK12_1290-1574722511,NaN,NaN,NaN,"Floor Day 6, pm",NaN,NaN,6175987,"No, patient refused",NaT,NaT,NaN


In [18]:
ICUSleep_DataTable[ICUSleep_DataTable.StudyID==3]

,StudyID,ADT_TransferIn,ADT_TransferOut,CAM_Date,BMFileConverted,BMDataAllGood,BMFileID,BMFileNotes,AirGoDone,AirGoFileNotes,EventName,ICU_Site,ICU_Bed,MRN,CAM_performed,ADT_Admission,ADT_Discharge,ADT_Bed
22,3,2018-06-18 01:12:00,2018-06-21 21:56:00,2018-06-18 15:30:00,1.0,NaN,BLK07_754-1529299617,NaN,NaN,NaN,"ICU Day 1, pm",Blake 7,54,1910753,Yes,2018-06-17 16:07:00,2018-06-28 14:53:00,B0754 A
23,3,2018-06-18 01:12:00,2018-06-21 21:56:00,2018-06-19 09:04:00,1.0,NaN,BLK07_754-1529299617,NaN,NaN,NaN,"ICU Day 2, am",Blake 7,54,1910753,Yes,2018-06-17 16:07:00,2018-06-28 14:53:00,B0754 A
24,3,2018-06-18 01:12:00,2018-06-21 21:56:00,2018-06-19 16:48:00,1.0,NaN,BLK07_754-1529299617,NaN,NaN,NaN,"ICU Day 2, pm",Blake 7,54,1910753,Yes,2018-06-17 16:07:00,2018-06-28 14:53:00,B0754 A
25,3,2018-06-18 01:12:00,2018-06-21 21:56:00,2018-06-20 08:37:00,1.0,NaN,BLK07_754-1529299617,NaN,NaN,NaN,"ICU Day 3, am",Blake 7,54,1910753,Yes,2018-06-17 16:07:00,2018-06-28 14:53:00,B0754 A
26,3,2018-06-18 01:12:00,2018-06-21 21:56:00,2018-06-20 17:00:00,1.0,NaN,BLK07_754-1529299617,NaN,NaN,NaN,"ICU Day 3, pm",Blake 7,54,1910753,Yes,2018-06-17 16:07:00,2018-06-28 14:53:00,B0754 A
27,3,2018-06-18 01:12:00,2018-06-21 21:56:00,2018-06-21 09:33:00,1.0,NaN,BLK07_754-1529299617,NaN,NaN,NaN,"ICU Day 4, am",Blake 7,54,1910753,Yes,2018-06-17 16:07:00,2018-06-28 14:53:00,B0754 A
28,3,2018-06-18 01:12:00,2018-06-21 21:56:00,2018-06-21 18:00:00,1.0,NaN,BLK07_754-1529299617,NaN,NaN,NaN,"ICU Day 4, pm",Blake 7,54,1910753,Yes,2018-06-17 16:07:00,2018-06-28 14:53:00,B0754 A
29,3,NaT,NaT,2018-06-22 10:03:00,NaN,NaN,BLK07_754-1529299617,NaN,NaN,NaN,"Floor Day 1, am",NaN,NaN,1910753,Yes,NaT,NaT,NaN
30,3,NaT,NaT,2018-06-22 18:00:00,NaN,NaN,BLK07_754-1529299617,NaN,NaN,NaN,"Floor Day 1, pm",NaN,NaN,1910753,Yes,NaT,NaT,NaN
31,3,NaT,NaT,2018-06-23 11:00:00,NaN,NaN,BLK07_754-1529299617,NaN,NaN,NaN,"Floor Day 2, am",NaN,NaN,1910753,Yes,NaT,NaT,NaN


In [18]:
# append conversion status. either new or in loop before.

###### get all fileIDs, needed currently for time fixing step in bedmaster.

In [19]:
# allFileIDs = (ICUSleep_DataTable.BMFileID.dropna()).values
# allFileIDs = np.unique(allFileIDs)
# allFileIDs.shape

### Step 2: Add ADT data to table

In [19]:
ADT = pd.read_csv(os.path.join(LogDir, 'ADT_cdacdb.csv'))

In [20]:
ADT.TransferInDTS = pd.to_datetime(ADT.TransferInDTS, infer_datetime_format=1)
ADT.TransferOutDTS = pd.to_datetime(ADT.TransferOutDTS, infer_datetime_format=1)
ADT.AdmissionDTS = pd.to_datetime(ADT.AdmissionDTS, infer_datetime_format=1)
ADT.DischargeDTS = pd.to_datetime(ADT.DischargeDTS, infer_datetime_format=1)

In [21]:
ADT.tail()

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS
747,5854072,1.563171e+09,1.565900e+09,1563388980,1.563572e+09,B1278,2019-07-17 13:43:00,2019-07-19 16:41:00,3260213550,MGH BLAKE 12 ICU,B1278 A,2019-07-15 01:13:00,2019-08-15 15:21:00
748,5854072,1.563171e+09,1.565900e+09,1563572460,1.565278e+09,G944,2019-07-19 16:41:00,2019-08-08 10:30:00,3260213550,MGH BIGELOW 9 MED,G0944 A,2019-07-15 01:13:00,2019-08-15 15:21:00
749,5854072,1.563171e+09,1.565900e+09,1565304180,1.565900e+09,G944,2019-08-08 17:43:00,2019-08-15 15:21:00,3260213550,MGH BIGELOW 9 MED,G0944 A,2019-07-15 01:13:00,2019-08-15 15:21:00
750,1750989,1.574766e+09,NaN,1575747300,1.576038e+09,B1260,2019-12-07 14:35:00,2019-12-10 23:19:00,3278402307,MGH BLAKE 12 ICU,B1260 A,2019-11-26 05:56:00,2020-01-01 00:00:00
751,1750989,1.574766e+09,NaN,1574825460,1.575747e+09,B1272,2019-11-26 22:31:00,2019-12-07 14:35:00,3278402307,MGH BLAKE 12 ICU,B1272 A,2019-11-26 05:56:00,2020-01-01 00:00:00


In [24]:
ADT[ADT.MRN==1910753]

,MRN,tAdmit,tDischarge,tStart,tEnd,stpLoc,TransferInDTS,TransferOutDTS,PatientEncounterID,DepartmentDSC,Bed,AdmissionDTS,DischargeDTS
192,1910753,1.484753e+09,1.485464e+09,1484777460,1.485464e+09,W1030,2017-01-18 17:11:00,2017-01-26 15:46:00,3142989191,MGH WHITE 10 MEDICINE,W1030 B,2017-01-18 10:21:00,2017-01-26 15:46:00
296,1910753,1.529270e+09,1.530216e+09,1529302320,1.529636e+09,B754,2018-06-18 01:12:00,2018-06-21 21:56:00,3205625962,MGH BLAKE 7 MICU,B0754 A,2018-06-17 16:07:00,2018-06-28 14:53:00


In [25]:
for studyID in np.unique(ICUSleep_DataTable.StudyID):
    
    MRN_sel = np.unique(ICUSleep_DataTable[ICUSleep_DataTable.StudyID == studyID].MRN)
    assert(MRN_sel.shape[0] == 1)
    MRN_sel = MRN_sel[0]
    
    assert(ADT[ADT.MRN == MRN_sel].TransferInDTS.shape[0] == ADT[ADT.MRN == MRN_sel].TransferOutDTS.shape[0])
    
    # loop through all Transfers in ADT data and add it to ICU sleep table:
    for iTransferEntry in range(ADT[ADT.MRN == MRN_sel].TransferInDTS.shape[0]):
        
        transferInSel = ADT[ADT.MRN == MRN_sel].TransferInDTS.iloc[iTransferEntry]
        transferOutSel = ADT[ADT.MRN == MRN_sel].TransferOutDTS.iloc[iTransferEntry]
        AdmissionSel = ADT[ADT.MRN == MRN_sel].AdmissionDTS.iloc[iTransferEntry]
        DischargeSel = ADT[ADT.MRN == MRN_sel].DischargeDTS.iloc[iTransferEntry]
        BedSel = ADT[ADT.MRN == MRN_sel].Bed.iloc[iTransferEntry]

        BooleanSelectionOfCAMWithinTransfer =(ICUSleep_DataTable.StudyID == studyID) & (ICUSleep_DataTable.CAM_Date >= transferInSel) & (ICUSleep_DataTable.CAM_Date <= transferOutSel )
        
        ICUSleep_DataTable.loc[BooleanSelectionOfCAMWithinTransfer, 'ADT_TransferIn'] = transferInSel 
        ICUSleep_DataTable.loc[BooleanSelectionOfCAMWithinTransfer, 'ADT_TransferOut'] = transferOutSel 
        ICUSleep_DataTable.loc[BooleanSelectionOfCAMWithinTransfer, 'ADT_Admission'] = AdmissionSel 
        ICUSleep_DataTable.loc[BooleanSelectionOfCAMWithinTransfer, 'ADT_Discharge'] = DischargeSel 
        ICUSleep_DataTable.loc[BooleanSelectionOfCAMWithinTransfer, 'ADT_Bed'] = BedSel 


    

#### STEP 3: 
- append conversion info ('StudyID','EventName','CAM_Date','CAM_performed') to ICUSleep_Table.  
overwrite all data. if something is manually changed, do that after this automatic step.

get conversion status of bedmaster files (done in FindNonConvertedFiles)

In [26]:
# LogDirTemp = 'C:/Users/wg984/Wolfgang/ICU-Sleep/'
filep=LogDir+'ConversionStatus.csv'
ConversionStatus = pd.read_csv(filep)
print(ConversionStatus.columns)
ConversionStatus = ConversionStatus[['studyID','fileID','Converted_V4','StartTime','EndTime','fileStartTime']]
ConversionStatus.columns = ['StudyID','BMfileID','BMFileConverted','StartTime','EndTime','FileStartTime']
ConversionStatus.StartTime = pd.to_datetime(ConversionStatus.StartTime, infer_datetime_format=1)
ConversionStatus.EndTime = pd.to_datetime(ConversionStatus.EndTime, infer_datetime_format=1)
ConversionStatus.FileStartTime = pd.to_datetime(ConversionStatus.FileStartTime, infer_datetime_format=1)

Index(['Converted_V4', 'studyID', 'fileID', 'StartTime', 'EndTime', 'stpIn',
       'tStart', 'tEnd', 'tDischarge', 'tAdmit', 'unixFileStartTime',
       'unixFileEndTime', 'fileStartTime', 'MRN', 'VisitIdentifier', 'grp',
       'ugid'],
      dtype='object')


In [27]:
ConversionStatus.StudyID = ConversionStatus.StudyID.astype('int')
ConversionStatus.tail()

,StudyID,BMfileID,BMFileConverted,StartTime,EndTime,FileStartTime
634,192,BLK12_1260-1575680691,1,2019-12-07 14:35:00,2019-12-10 23:19:00,2019-12-07 01:04:51
635,192,BLK12_1260-1575747932,1,2019-12-07 14:35:00,2019-12-10 23:19:00,2019-12-07 19:45:32
636,192,BLK12_1272-1574812978,1,2019-11-26 22:31:00,2019-12-07 14:35:00,2019-11-27 00:02:58
637,193,BLK12_1252-1574537100,1,2019-11-23 14:54:00,2019-11-29 12:38:00,2019-11-23 19:25:00
638,194,BLK12_1290-1574722511,1,2019-11-25 18:31:00,2019-11-27 18:03:00,2019-11-25 22:55:11


In [42]:
ConversionStatus[ConversionStatus.StudyID==3]

,StudyID,BMfileID,BMFileConverted,StartTime,EndTime,FileStartTime
4,3,BLK07_754-1529299617,1,2018-06-18 01:12:00,2018-06-21 21:56:00,2018-06-18 05:26:57


In [29]:
ICUSleep_DataTable[ICUSleep_DataTable.StudyID==151].CAM_Date.min()

Timestamp('2019-09-10 15:50:00')

In [30]:
ICUSleep_DataTable[ICUSleep_DataTable.StudyID==151].CAM_Date.max()

Timestamp('2019-09-18 16:15:00')

In [31]:
ConversionStatus[ConversionStatus.BMfileID == 'BIG13_1346-1525106475']

,StudyID,BMfileID,BMFileConverted,StartTime,EndTime,FileStartTime
516,151,BIG13_1346-1525106475,1,2018-04-24 17:15:00,2018-04-30 11:45:00,2018-04-30 16:41:15


In [41]:
### this if multiple bedmaster files exhibit same startTime endTime, i.e. multiple BM file for a stay of patient in ICU. both files needed!

if 0:
    for study_id in pd.unique(ConversionStatus.StudyID):
        bm_files_subject = ConversionStatus[ConversionStatus.StudyID==study_id]
        if bm_files_subject.StartTime.apply(lambda x: x.date()).duplicated().sum() > 0 or bm_files_subject.EndTime.apply(lambda x: x.date()).duplicated().sum() > 0:
    #         print(study_id)
            print(bm_files_subject)


In [33]:
# for row_idx in range(ICUSleep_DataTable.shape[0]):
    
#     DataTable_Sel = ICUSleep_DataTable.iloc[row_idx:]
#     ConversionStatus_Sel = ConversionStatus.loc[ConversionStatus.StudyID == study_id]
#     ConversionStatus_Sel.sort_values(by='FileStartTime', inplace=True)

#     min_date = DataTable_Sel.CAM_Date.min() - timedelta(days=3)
#     max_date = DataTable_Sel.CAM_Date.max() + timedelta(days=3)

#     start_within_stay = np.logical_and(ConversionStatus_Sel.StartTime > min_date, ConversionStatus_Sel.StartTime < max_date).values
#     end_within_stay = np.logical_and(ConversionStatus_Sel.EndTime > min_date, ConversionStatus_Sel.EndTime < max_date).values
#     full_stay = np.logical_and(ConversionStatus_Sel.StartTime < min_date, ConversionStatus_Sel.EndTime > max_date).values
#     associated_to_stay = np.logical_or(start_within_stay, end_within_stay, full_stay)
# #     print(associated_to_stay)
#     file_ids_associated_to_stay = pd.unique(ConversionStatus_Sel.BMfileID[associated_to_stay])
# #     print(file_ids_associated_to_stay)
#     ICUSleep_DataTable.loc[ICUSleep_DataTable.StudyID == study_id,'BMFileID'] = ','.join(file_ids_associated_to_stay)


In [34]:
# for study_id in np.unique(ICUSleep_DataTable.StudyID):
    
#     DataTable_Sel = ICUSleep_DataTable.loc[ICUSleep_DataTable.StudyID == study_id]
#     ConversionStatus_Sel = ConversionStatus.loc[ConversionStatus.StudyID == study_id]
#     ConversionStatus_Sel.sort_values(by='FileStartTime', inplace=True)

#     min_date = DataTable_Sel.CAM_Date.min() - timedelta(days=3)
#     max_date = DataTable_Sel.CAM_Date.max() + timedelta(days=3)

#     start_within_stay = np.logical_and(ConversionStatus_Sel.StartTime > min_date, ConversionStatus_Sel.StartTime < max_date).values
#     end_within_stay = np.logical_and(ConversionStatus_Sel.EndTime > min_date, ConversionStatus_Sel.EndTime < max_date).values
#     full_stay = np.logical_and(ConversionStatus_Sel.StartTime < min_date, ConversionStatus_Sel.EndTime > max_date).values
#     associated_to_stay = np.logical_or(start_within_stay, end_within_stay, full_stay)
# #     print(associated_to_stay)
#     file_ids_associated_to_stay = pd.unique(ConversionStatus_Sel.BMfileID[associated_to_stay])
# #     print(file_ids_associated_to_stay)
#     ICUSleep_DataTable.loc[ICUSleep_DataTable.StudyID == study_id,'BMFileID'] = ','.join(file_ids_associated_to_stay)


In [35]:
# for rowIdx in range(ICUSleep_DataTable.shape[0]):
    
#     DataTable_Sel = ICUSleep_DataTable.iloc[rowIdx,:]
#     ConversionStatus_Sel = ConversionStatus.loc[ConversionStatus.StudyID == DataTable_Sel.StudyID]
#     ConversionStatus_Sel.sort_values(by='FileStartTime', inplace=True)
    
#     # can be more than 1 file:
#     for ConvIdx in range(ConversionStatus_Sel.shape[0]):
#         Start_Tmp = ConversionStatus_Sel.StartTime.iloc[ConvIdx]
#         End_Tmp = ConversionStatus_Sel.EndTime.iloc[ConvIdx]
        
#         min_date = DataTable_Sel.ADT_TransferIn
#         max_date = DataTable_Sel.ADT_TransferOut

#         start_within_stay = np.logical_and(Start_Tmp > min_date, Start_Tmp < max_date).values
#         end_within_stay = np.logical_and(End_Tmp > min_date, End_Tmp < max_date).values
#         full_stay = np.logical_and(Start_Tmp < min_date, End_Tmp > max_date).values
#         associated_to_stay = np.logical_or(start_within_stay, end_within_stay, full_stay)
#     #     print(associated_to_stay)
# #         file_ids_associated_to_stay = pd.unique(ConversionStatus_Sel.BMfileID[associated_to_stay])
#     #     print(file_ids_associated_to_stay)
# #         ICUSleep_DataTable.loc[ICUSleep_DataTable.StudyID == study_id,'BMFileID'] = ','.join(file_ids_associated_to_stay)

# #         if (Start_Tmp <= DataTable_Sel.CAM_Date) & (End_Tmp >= DataTable_Sel.CAM_Date):
#         if associated_to_stay:
#             if not pd.isna(ICUSleep_DataTable.BMFileID.iloc[rowIdx]):  # second bm file for this date:
#                 if ConversionStatus_Sel.BMfileID.iloc[ConvIdx] in ICUSleep_DataTable.BMFileID.iloc[rowIdx]: continue # do not append same fileID again.
#                 ICUSleep_DataTable.BMFileID.iloc[rowIdx] += ',' + ConversionStatus_Sel.BMfileID.iloc[ConvIdx]
#                 ICUSleep_DataTable.BMFileConverted.iloc[rowIdx] = min(ICUSleep_DataTable.BMFileConverted.iloc[rowIdx],ConversionStatus_Sel.BMFileConverted.iloc[ConvIdx])
#             else: # first bm file for this date   
#                 ICUSleep_DataTable.BMFileID.iloc[rowIdx] = ConversionStatus_Sel.BMfileID.iloc[ConvIdx]
#                 ICUSleep_DataTable.BMFileConverted.iloc[rowIdx] = ConversionStatus_Sel.BMFileConverted.iloc[ConvIdx]

    
# # duplicateFileIDs = ConversionStatus_Sel[ConversionStatus_Sel.BMfileID.duplicated()].BMfileID.values

# # for duplicateFileID in duplicateFileIDs:
# #     minStartTime = min(ConversionStatus_Sel.StartTime)
# #     maxEndTime = max(ConversionStatus_Sel.EndTime)
# #     BMFileConverted = min(ConversionStatus_Sel.BMFileConverted)





In [36]:
# rowIdx= 395
# DataTable_Sel = ICUSleep_DataTable.iloc[rowIdx,:]
# ConversionStatus_Sel = ConversionStatus.loc[ConversionStatus.StudyID == DataTable_Sel.StudyID]
# ConversionStatus_Sel.sort_values(by='FileStartTime', inplace=True)

In [37]:
for rowIdx in ICUSleep_DataTable.index:
    
    DataTable_Sel = ICUSleep_DataTable.loc[rowIdx,:]
    ConversionStatus_Sel = ConversionStatus.loc[ConversionStatus.StudyID == DataTable_Sel.StudyID]
    ConversionStatus_Sel.sort_values(by='FileStartTime', inplace=True)
    
    # can be more than 1 file:
    for ConvIdx in range(ConversionStatus_Sel.shape[0]):
        Start_Tmp = ConversionStatus_Sel.StartTime.iloc[ConvIdx]
        End_Tmp = ConversionStatus_Sel.EndTime.iloc[ConvIdx]
        
        min_date = DataTable_Sel.ADT_TransferIn
        max_date = DataTable_Sel.ADT_TransferOut

        start_within_stay = np.logical_and(Start_Tmp >= min_date, Start_Tmp <= max_date)
        end_within_stay = np.logical_and(End_Tmp >= min_date, End_Tmp <= max_date)
        full_stay = np.logical_and(Start_Tmp <= min_date, End_Tmp >= max_date)
        associated_to_stay = any([start_within_stay, end_within_stay, full_stay])

        #     print(associated_to_stay)
#         file_ids_associated_to_stay = pd.unique(ConversionStatus_Sel.BMfileID[associated_to_stay])
    #     print(file_ids_associated_to_stay)
#         ICUSleep_DataTable.loc[ICUSleep_DataTable.StudyID == study_id,'BMFileID'] = ','.join(file_ids_associated_to_stay)

#         if (Start_Tmp <= DataTable_Sel.CAM_Date) & (End_Tmp >= DataTable_Sel.CAM_Date):

#         print(Start_Tmp)
#         print(End_Tmp)
#         print(min_date)
#         print(max_date)
#         print('\n')
        
        if associated_to_stay:
            if not pd.isna(ICUSleep_DataTable.BMFileID.loc[rowIdx]):  # second bm file for this date:
                if ConversionStatus_Sel.BMfileID.iloc[ConvIdx] in ICUSleep_DataTable.BMFileID.loc[rowIdx]: continue # do not append same fileID again.
                ICUSleep_DataTable.loc[rowIdx,'BMFileID'] += ',' + ConversionStatus_Sel.BMfileID.iloc[ConvIdx]
                ICUSleep_DataTable.loc[rowIdx,'BMFileConverted'] = min(ICUSleep_DataTable.BMFileConverted.loc[rowIdx],ConversionStatus_Sel.BMFileConverted.iloc[ConvIdx])
            else: # first bm file for this date   
                ICUSleep_DataTable.loc[rowIdx,'BMFileID'] = ConversionStatus_Sel.BMfileID.iloc[ConvIdx]
                ICUSleep_DataTable.loc[rowIdx,'BMFileConverted'] = ConversionStatus_Sel.BMFileConverted.iloc[ConvIdx]

C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """


In [38]:
def insert_BMFileNote(studyID, EventName, Note, BMFileConverted = np.nan):
    ICUSleep_DataTable.loc[(ICUSleep_DataTable.StudyID == studyID) & (ICUSleep_DataTable.EventName == EventName),'BMFileNotes'] = Note
    if BMFileConverted is not np.nan:
        ICUSleep_DataTable.loc[(ICUSleep_DataTable.StudyID == studyID) & (ICUSleep_DataTable.EventName == EventName),'BMFileConverted'] = BMFileConverted

In [39]:
insert_BMFileNote(4, 'ICU Day 4, pm', 'No BM Data for parts of this afternoon',2)
insert_BMFileNote(7, 'ICU Day 3, pm', 'No BM Data for parts of this afternoon',2)
insert_BMFileNote(16, 'ICU Day 2, pm', 'No BM Data, Patient got discharged around this time',2)
insert_BMFileNote(17, 'ICU Day 3, pm', 'No BM Data for parts of this afternoon',2)
insert_BMFileNote(17, 'ICU Day 6, am', 'No BM Data for parts of this morning',2)
insert_BMFileNote(43, 'ICU Day 2, pm', 'No BM Data for parts of this afternoon',2)
insert_BMFileNote(47, 'ICU Day 3, pm', 'No BM Data, Patient got discharged around this time',2)
insert_BMFileNote(51, 'ICU Day 2, pm', 'TOCHECK. ADT CDAC stops at 03-15.',2)
insert_BMFileNote(52, 'ICU Day 2, am', 'No BM Data for parts of this morning',2)
insert_BMFileNote(54, 'ICU Day 3, pm', 'No BM Data, Patient got discharged around this time',2)
insert_BMFileNote(60, 'ICU Day 3, pm', 'No BM Data, Patient got discharged around this time',2)
insert_BMFileNote(74, 'ICU Day 4, pm', 'No BM Data, Patient got discharged around this time',2)
insert_BMFileNote(76, 'ICU Day 4, pm', 'No BM Data for parts of this afternoon',2)
insert_BMFileNote(76, 'ICU Day 7, am', 'No BM Data for parts of this morning',2)

In [40]:
# for _ICUonly only keep ICU dates:
ICUSleep_DataTable_ICUonly = ICUSleep_DataTable.where(ICUSleep_DataTable.EventName.str.contains('ICU')).dropna(how='all')

In [40]:
ICUSleep_DataTable.to_csv(os.path.join(LogDir,'ICUSleep_DataTable.csv'), index=False)
ICUSleep_DataTable_ICUonly.to_csv(os.path.join(LogDir,'ICUSleep_DataTable_ICUonly.csv'), index=False)

# also add same tables to GitHub for backup:
# ICUSleep_DataTable.to_csv(os.path.join(GitHubBackupDir,'ICUSleep_DataTable.csv'), index=False)
# ICUSleep_DataTable_ICUonly.to_csv(os.path.join(GitHubBackupDir,'ICUSleep_DataTable_ICUonly.csv'), index=False)

In [41]:
ICUSleep_DataTable_ICU_OnlyNonConverted = ICUSleep_DataTable_ICUonly[(ICUSleep_DataTable_ICUonly.BMFileConverted!=1)]
ICUSleep_DataTable_ICU_OnlyNonConverted

,StudyID,ADT_TransferIn,ADT_TransferOut,CAM_Date,BMFileConverted,BMDataAllGood,BMFileID,BMFileNotes,AirGoDone,AirGoFileNotes,EventName,ICU_Site,ICU_Bed,MRN,CAM_performed,ADT_Admission,ADT_Discharge,ADT_Bed
48,4.0,NaT,NaT,2018-06-22 13:04:00,2.0,NaN,"BLK07_788-1529370699,LUN07_778-1529875345",No BM Data for parts of this afternoon,NaN,NaN,"ICU Day 4, pm",Blake 7,88,6188100.0,"No, patient unavailable",NaT,NaT,NaN
102,7.0,NaT,NaT,2018-08-09 19:33:00,2.0,NaN,BLK12_1260-1533560479,No BM Data for parts of this afternoon,NaN,NaN,"ICU Day 3, pm",Blake 12,60,6420799.0,"No, patient unavailable",NaT,NaT,NaN
293,16.0,NaT,NaT,2018-10-16 18:30:00,NaN,NaN,ELL04_438-1539459164,NaN,NaN,NaN,"ICU Day 2, pm",Ellison 4,38,1902538.0,Yes,NaT,NaT,NaN
312,17.0,2018-10-16 18:08:00,2018-10-23 09:01:00,2018-10-20 16:53:00,2.0,NaN,BLK12_1288-1539370921,No BM Data for parts of this afternoon,NaN,NaN,"ICU Day 3, pm",Blake 12,88,6479611.0,"No, patient unavailable",2018-10-13 20:35:00,2018-11-27 12:47:00,B1288 A
317,17.0,NaT,NaT,2018-10-23 10:29:00,2.0,NaN,BLK12_1288-1539370921,No BM Data for parts of this morning,NaN,NaN,"ICU Day 6, am",Blake 12,88,6479611.0,"No, patient unavailable",NaT,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2681,192.0,2019-11-26 22:31:00,2019-12-07 14:35:00,2019-11-29 11:50:00,0.0,NaN,"BLK12_1272-1574812978,BLK12_1260-1575680691,BL...",NaN,NaN,NaN,"ICU Day 3, am",Blake 12,72,1750989.0,"No, patient unavailable",2019-11-26 05:56:00,2020-01-01 00:00:00,B1272 A
2683,192.0,2019-11-26 22:31:00,2019-12-07 14:35:00,2019-12-01 10:31:00,0.0,NaN,"BLK12_1272-1574812978,BLK12_1260-1575680691,BL...",NaN,NaN,NaN,"ICU Day 4, am",Blake 12,72,1750989.0,"No, patient refused",2019-11-26 05:56:00,2020-01-01 00:00:00,B1272 A
2684,192.0,2019-11-26 22:31:00,2019-12-07 14:35:00,2019-12-01 10:32:00,0.0,NaN,"BLK12_1272-1574812978,BLK12_1260-1575680691,BL...",NaN,NaN,NaN,"ICU Day 5, am",Blake 12,72,1750989.0,"No, patient refused",2019-11-26 05:56:00,2020-01-01 00:00:00,B1272 A
2685,192.0,2019-11-26 22:31:00,2019-12-07 14:35:00,2019-12-01 18:39:00,0.0,NaN,"BLK12_1272-1574812978,BLK12_1260-1575680691,BL...",NaN,NaN,NaN,"ICU Day 4, pm",Blake 12,72,1750989.0,"No, patient refused",2019-11-26 05:56:00,2020-01-01 00:00:00,B1272 A


In [20]:
ICUSleep_DataTable_ICU_OnlyNonConverted.to_csv(os.path.join(LogDirTemp,'ICUSleep_DataTable_ICU_OnlyNonConverted.csv'), index=False)

In [21]:
# subject 76: BLK12_1282-1556320216 is not converted. because of wrong time info of other fileID this does not show  up here yet.